# Stroke Prediction

## Importing necessary libraries

In [1]:
import numpy as np
import pandas as pd 

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
import optuna
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, fbeta_score, make_scorer

## Import dataset

In [2]:
df = pd.read_csv('../dataset/healthcare-dataset-stroke-data.csv')

In [3]:
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


In [5]:
df.describe()

,id,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,5110.000000,5110.000000,5110.000000,5110.000000,5110.000000,4909.000000,5110.000000
mean,36517.829354,43.226614,0.097456,0.054012,106.147677,28.893237,0.048728
std,21161.721625,22.612647,0.296607,0.226063,45.283560,7.854067,0.215320
min,67.000000,0.080000,0.000000,0.000000,55.120000,10.300000,0.000000
25%,17741.250000,25.000000,0.000000,0.000000,77.245000,23.500000,0.000000
50%,36932.000000,45.000000,0.000000,0.000000,91.885000,28.100000,0.000000
75%,54682.000000,61.000000,0.000000,0.000000,114.090000,33.100000,0.000000
max,72940.000000,82.000000,1.000000,1.000000,271.740000,97.600000,1.000000


In [6]:
df.describe(include=[object])

,gender,ever_married,work_type,Residence_type,smoking_status
count,5110,5110,5110,5110,5110
unique,3,2,5,2,4
top,Female,Yes,Private,Urban,never smoked
freq,2994,3353,2925,2596,1892


## Rename column for consistent formatting of name

In [7]:
df.rename(columns={'Residence_type': 'residence_type'}, inplace=True)

In [8]:
pd.set_option('display.max_columns', None)

In [9]:
pd.set_option('display.max_rows', None)

### Impute missing values in BMI column
For 201 entries missing BMI values, these entries will be imputed with the median BMI value

In [10]:
median_bmi = df['bmi'].median()
df['bmi'].fillna(median_bmi, inplace=True)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16644\2337012406.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bmi'].fillna(median_bmi, inplace=True)


In [11]:
print(df['bmi'].isna().sum())

0


## Feature Engineering

### Converting categorical columns to numerical columns

In [12]:
df['male'] = (df['gender'] == 'Male').astype(int)

In [13]:
df['ever_married'] = (df['ever_married'] == 'Yes').astype(int)

In [14]:
df['urban'] = (df['residence_type'] == 'Urban').astype(int)

In [15]:
df['smoke'] = ((df['smoking_status'] == 'formerly smoked') | (df['smoking_status'] == 'smokes')).astype(int)

In [16]:
df['have_work'] = ((df['work_type'] == 'Govt_job') | (df['work_type'] == 'Private') | (df['work_type'] == 'Self-employed')).astype(int)

In [17]:
df['hypertension_or_heart_disease'] = df['hypertension'] | df['heart_disease']

In [18]:
df = pd.get_dummies(df, columns=['work_type', 'smoking_status'], drop_first = False)

### Extra column for the age of married people

In [19]:
df['age_ever_married'] = df['age'] * df['ever_married']

### Creating columns representing outstanding values for BMI and Average Glucose Level

In [20]:
df['obese'] = df['bmi'] >= 30

In [21]:
df['high_avg_glucose_level'] = df['avg_glucose_level'] >= 150

### Converting boolean columns to numerical

In [22]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

### Reformatting column names

In [23]:
df.rename(columns={'work_type_Govt_job': 'work_type_govt_job', 'work_type_Never_worked': 'work_type_never_worked', 'work_type_Private': 'work_type_private', 'work_type_Self-employed': 'work_type_self_employed'}, inplace=True)

In [24]:
df.rename(columns={'smoking_status_Unknown': 'smoking_status_unknown', 'smoking_status_formerly smoked': 'smoking_status_formerly_smoked', 'smoking_status_never smoked': 'smoking_status_never_smoked'}, inplace=True)

In [25]:
df.drop(columns=['gender', 'residence_type'], inplace=True)

### Log Transforming BMI and Average Glucose Level columns

In [26]:
df['log_bmi'] = np.log(df['bmi'])

In [27]:
df['log_avg_glucose_level'] = np.log(df['avg_glucose_level'])

In [28]:
df.head()

,id,age,hypertension,heart_disease,ever_married,avg_glucose_level,bmi,stroke,male,urban,smoke,have_work,hypertension_or_heart_disease,work_type_govt_job,work_type_never_worked,work_type_private,work_type_self_employed,work_type_children,smoking_status_unknown,smoking_status_formerly_smoked,smoking_status_never_smoked,smoking_status_smokes,age_ever_married,obese,high_avg_glucose_level,log_bmi,log_avg_glucose_level
0,9046,67.0,0,1,1,228.69,36.6,1,1,1,1,1,1,0,0,1,0,0,0,1,0,0,67.0,1,1,3.600048,5.432367
1,51676,61.0,0,0,1,202.21,28.1,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,61.0,0,1,3.335770,5.309307
2,31112,80.0,0,1,1,105.92,32.5,1,1,0,0,1,1,0,0,1,0,0,0,0,1,0,80.0,1,0,3.481240,4.662684
3,60182,49.0,0,0,1,171.23,34.4,1,0,1,1,1,0,0,0,1,0,0,0,0,0,1,49.0,1,1,3.538057,5.143008
4,1665,79.0,1,0,1,174.12,24.0,1,0,0,0,1,1,0,0,0,1,0,0,0,1,0,79.0,0,1,3.178054,5.159745


In [29]:
df.shape

(5110, 27)

## Model Building

### Splitting to training, validation, and testing sets

In [30]:
columns_name = columns_name = [col for col in df.columns if col != 'id']

In [31]:
X = df[columns_name]
X.drop('stroke', axis=1, inplace=True)
y = df['stroke']

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_16644\91966498.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop('stroke', axis=1, inplace=True)


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5, random_state = 42, stratify = y_test)

### Random Oversampling on Training Data

In [33]:
resampler = RandomOverSampler()

In [34]:
X_train_res, y_train_res = resampler.fit_resample(X_train, y_train)

### Create base random forest classifier model

In [35]:
base_classifier = lgb.LGBMClassifier(random_state=42, verbose=-1)

### Create grid for hyperparameter tuning

In [36]:
param_grid_lgbm = {
    'num_leaves': [15, 31, 63, 127],
    'max_depth': [3, 5, 7, 10, -1], 
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200, 300],
    'min_child_samples': [5, 10, 20, 30],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1], 
    'reg_lambda': [0, 0.1, 0.5, 1],  
    'scale_pos_weight': [1, 3, 5, 10, 15, 20]  
}

In [37]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [38]:
grid_lgbm = GridSearchCV(
    estimator=base_classifier,
    param_grid=param_grid_lgbm,
    scoring=f2_scorer,
    cv=5,
    verbose=2,
    n_jobs=-1
)

In [39]:
# importance_df = pd.DataFrame({'Feature': X_train_res.columns, 'Importance': base_classifier.feature_importances_})
# importance_df = importance_df.sort_values(by='Importance', ascending=False)
# print(importance_df)

### Fitting base classifier on unsampled data

In [40]:
base_classifier.fit(X_train_res, y_train_res)

LGBMClassifier(random_state=42, verbose=-1)

In [41]:
importances = base_classifier.feature_importances_

In [42]:
feature_imp_df = pd.DataFrame({'Feature':[col for col in X_train], 'Importance': importances}).sort_values(
    'Importance', ascending=False)

In [43]:
print(feature_imp_df)

                           Feature  Importance
4                avg_glucose_level         887
5                              bmi         794
0                              age         492
20                age_ever_married         201
7                            urban          66
6                             male          64
1                     hypertension          61
14         work_type_self_employed          60
13               work_type_private          56
18     smoking_status_never_smoked          56
10   hypertension_or_heart_disease          48
19           smoking_status_smokes          46
2                    heart_disease          41
16          smoking_status_unknown          32
8                            smoke          31
17  smoking_status_formerly_smoked          27
11              work_type_govt_job          19
3                     ever_married          17
21                           obese           1
22          high_avg_glucose_level           1
23           

In [44]:
retained_columns = ['avg_glucose_level', 'bmi', 'age', 'urban', 'male', 'work_type_self_employed', 'work_type_private', 'smoking_status_never_smoked', 'smoking_status_smokes', 'heart_disease']

In [45]:
X_train_res_retained = X_train_res[retained_columns]

In [46]:
X_val_retained = X_val[retained_columns]

In [47]:
MIN_PRECISION = 0.15

In [48]:
def objective_lgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 1),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1, 20),
        'random_state': 42,
        'verbose': -1
    }
    
    model = lgb.LGBMClassifier(**params)
    
    model.fit(X_train_res_retained, y_train_res)
    
    y_pred_val = model.predict(X_val_retained)
    
    recall = recall_score(y_val, y_pred_val)
    precision = precision_score(y_val, y_pred_val)
    
    if precision < MIN_PRECISION:
        return 0
    return recall + (precision * 0.01)

In [49]:
# grid_lgbm.fit(X_train_res_retained, y_train_res)

In [50]:
study_lgbm = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_lgbm.optimize(objective_lgbm, n_trials=100, show_progress_bar=True)

[I 2026-01-19 14:11:17,349] A new study created in memory with name: no-name-d85f6fad-1000-466f-b7fe-1fdb06ffad88


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-01-19 14:11:18,519] Trial 0 finished with value: 0.2996886016451234 and parameters: {'num_leaves': 57, 'max_depth': 15, 'learning_rate': 0.1205712628744377, 'n_estimators': 200, 'min_child_samples': 12, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.8661761457749352, 'reg_lambda': 0.6011150117432088, 'scale_pos_weight': 14.453378978124864}. Best is trial 0 with value: 0.2996886016451234.
[I 2026-01-19 14:11:19,210] Trial 1 finished with value: 0.48805170387779084 and parameters: {'num_leaves': 17, 'max_depth': 15, 'learning_rate': 0.16967533607196555, 'n_estimators': 103, 'min_child_samples': 13, 'subsample': 0.5917022549267169, 'colsample_bytree': 0.6521211214797689, 'reg_alpha': 0.5247564316322378, 'reg_lambda': 0.43194501864211576, 'scale_pos_weight': 6.533353663762797}. Best is trial 1 with value: 0.48805170387779084.
[I 2026-01-19 14:11:19,435] Trial 2 finished with value: 0.0 and parameters: {'num_leaves': 84, 'max_depth': 4, 'lear

In [51]:
# best_params_lgbm = grid_lgbm.best_params_

In [52]:
best_params_lgbm = study_lgbm.best_params

In [53]:
print(best_params_lgbm)

{'num_leaves': 30, 'max_depth': 14, 'learning_rate': 0.019035163813321078, 'n_estimators': 205, 'min_child_samples': 14, 'subsample': 0.6717150015850915, 'colsample_bytree': 0.6841941179270401, 'reg_alpha': 0.6788318683843165, 'reg_lambda': 0.24634858160961537, 'scale_pos_weight': 6.490382511116681}


In [54]:
final_classifier = lgb.LGBMClassifier(**best_params_lgbm, random_state=42, verbose=-1)

In [55]:
final_classifier.fit(X_train_res_retained, y_train_res)

LGBMClassifier(colsample_bytree=0.6841941179270401,
               learning_rate=0.019035163813321078, max_depth=14,
               min_child_samples=14, n_estimators=205, num_leaves=30,
               random_state=42, reg_alpha=0.6788318683843165,
               reg_lambda=0.24634858160961537,
               scale_pos_weight=6.490382511116681, subsample=0.6717150015850915,
               verbose=-1)

In [56]:
y_pred_val = final_classifier.predict(X_val_retained)

In [57]:
y_proba_val = final_classifier.predict_proba(X_val_retained)[:, 1]

In [58]:
print(confusion_matrix(y_val, y_pred_val))

[[573 156]
 [  9  28]]


In [59]:
print(accuracy_score(y_val, y_pred_val))

0.7845953002610966


In [60]:
print(precision_score(y_val, y_pred_val))

0.15217391304347827


In [61]:
print(recall_score(y_val, y_pred_val))

0.7567567567567568


In [62]:
print(fbeta_score(y_val, y_pred_val, beta=2))

0.42168674698795183


### Threshold tuning

In [63]:
best_threshold = 0.5

In [64]:
best_precision_score = 0

In [65]:
best_recall_score = 0

In [66]:
best_f2_score = 0

In [67]:
for threshold in np.arange(0.1, 0.95, 0.05):
    y_pred_val_threshold = (y_proba_val >= threshold).astype(int)
    recall = recall_score(y_val, y_pred_val_threshold)
    precision = precision_score(y_val, y_pred_val_threshold)
    f2_score = fbeta_score(y_val, y_pred_val_threshold, beta=2)
    
    print(f'Threshold {threshold} has recall score {f2_score}')
    
    if precision >= 0.15:
        current_recall_score = recall
        if current_recall_score > best_recall_score:
            best_recall_score = current_recall_score
            best_precision_score = precision
            best_threshold = threshold
        elif current_recall_score == best_recall_score and precision > best_precision_score:
            best_precision_score = precision
            best_threshold = threshold
        
        # current_f2_score = f2_score
        # if current_f2_score > best_f2_score:
        #     best_f2_score = current_f2_score
        #     best_threshold = threshold 

Threshold 0.1 has recall score 0.3333333333333333
Threshold 0.15000000000000002 has recall score 0.3610503282275711
Threshold 0.20000000000000004 has recall score 0.35962877030162416
Threshold 0.25000000000000006 has recall score 0.3588516746411483
Threshold 0.30000000000000004 has recall score 0.3778337531486146
Threshold 0.3500000000000001 has recall score 0.3815789473684211
Threshold 0.40000000000000013 has recall score 0.4027777777777778
Threshold 0.45000000000000007 has recall score 0.42151162790697677
Threshold 0.5000000000000001 has recall score 0.42168674698795183
Threshold 0.5500000000000002 has recall score 0.43478260869565216
Threshold 0.6000000000000002 has recall score 0.4340836012861736
Threshold 0.6500000000000001 has recall score 0.4426229508196721
Threshold 0.7000000000000002 has recall score 0.44982698961937717
Threshold 0.7500000000000002 has recall score 0.44964028776978415
Threshold 0.8000000000000002 has recall score 0.44573643410852715
Threshold 0.850000000000000

In [68]:
print(best_threshold)

0.5500000000000002


In [69]:
print(best_recall_score)

0.7567567567567568


### Retrain on training + validation set

In [70]:
X_train_full = pd.concat([X_train_res_retained, X_val_retained], axis=0, ignore_index=True)
y_train_full = pd.concat([y_train_res, y_val], axis=0, ignore_index=True)

In [71]:
final_classifier.fit(X_train_full, y_train_full)

LGBMClassifier(colsample_bytree=0.6841941179270401,
               learning_rate=0.019035163813321078, max_depth=14,
               min_child_samples=14, n_estimators=205, num_leaves=30,
               random_state=42, reg_alpha=0.6788318683843165,
               reg_lambda=0.24634858160961537,
               scale_pos_weight=6.490382511116681, subsample=0.6717150015850915,
               verbose=-1)

### Evaluate on test set

In [72]:
X_test_retained = X_test[retained_columns]

In [73]:
y_pred_test = final_classifier.predict(X_test_retained)

In [74]:
y_proba_test = final_classifier.predict_proba(X_test_retained)[:, 1]

In [75]:
print(confusion_matrix(y_test, y_pred_test))

[[542 187]
 [ 12  26]]


In [76]:
print(accuracy_score(y_test, y_pred_test))

0.7405475880052151


In [77]:
print(precision_score(y_test, y_pred_test))

0.12206572769953052


In [78]:
print(recall_score(y_test, y_pred_test))

0.6842105263157895


In [79]:
print(fbeta_score(y_test, y_pred_test, beta=2))

0.3561643835616438


### Test with best threshold 0.55

In [80]:
y_pred_test_threshold = (y_proba_test >= best_threshold).astype(int)

In [81]:
print(confusion_matrix(y_test, y_pred_test_threshold))

[[558 171]
 [ 12  26]]


In [82]:
print(accuracy_score(y_test, y_pred_test_threshold))

0.7614080834419817


In [83]:
print(precision_score(y_test, y_pred_test_threshold))

0.1319796954314721


In [84]:
print(recall_score(y_test, y_pred_test_threshold))

0.6842105263157895


In [85]:
print(fbeta_score(y_test, y_pred_test_threshold, beta=2))

0.37249283667621774
